In [3]:
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
text = ""

with pdfplumber.open(r"C:\Users\aishw\Desktop\Learning\Projects\Resume Project\Pushkar_s_Resume.pdf") as pdf:
    for page in pdf.pages:
        extracted = page.extract_text()

        if extracted:
            text += extracted

print(text[:1000])

Pushkar Chhokar
(cid:131) +91-7389530419 # pushkarcd123@gmail.com (cid:239) pushkar-chhokar § github.com/pushkarsingh26
Career Objective
Aspiring AI & Generative AI Engineer with a strong foundation in Python, LLMs, embeddings, vector databases, and
RAG pipelines. Seeking an internship to build scalable AI-powered applications, contribute to real-world GenAI
projects, and grow through hands-on engineering experience.
Education
Acropolis Institute of Technology and Research Indore, M.P.
B.Tech – Computer Science and Engineering | CGPA: 6.68 Nov 2023 – Jun 2027
New Eklavya English H.S. School Suvasra, Mandsaur, M.P.
Class XII | M.P. Board, Bhopal | Percentage: 88.4% 2023
Suraj Convent H.S. School Rajod, Dhar, M.P.
Class X | M.P. Board, Bhopal | Percentage: 90% 2021
Projects
IntelliDocs-RAG | Python, LangChain, FAISS, OpenAI API, FastAPI | Code Ongoing
– Built an end-to-end RAG pipeline allowing users to upload documents and query them in natural language with
high contextual accuracy.
– 

In [5]:
import re

text = re.sub(r'\(cid:\d+\)', '', text)
text = text.replace('\n', ' ')
text = re.sub(r'\s+', ' ', text)

In [53]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_text(text)

print("Total Chunks:", len(chunks))

Total Chunks: 9


In [31]:
print(chunks[0])

Pushkar Chhokar +91-7389530419 # pushkarcd123@gmail.com pushkar-chhokar § github.com/pushkarsingh26 Career Objective Aspiring AI & Generative AI Engineer with a strong foundation in Python, LLMs, embeddings, vector databases, and RAG pipelines. Seeking an internship to build scalable AI-powered applications, contribute to real-world GenAI projects, and grow through hands-on engineering experience


In [32]:
from dataclasses import dataclass
from typing import List

import numpy as np
from sentence_transformers import SentenceTransformer


@dataclass
class SearchResult:
    page_content: str
    score: float


class EmbeddingManager:
    """Handles embedding generation using sentence-transformers"""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully, embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


class SimpleVectorStore:
    def __init__(self, texts: List[str], embedding_manager: EmbeddingManager):
        self.texts = texts
        self.embedding_manager = embedding_manager
        self.embeddings = self.embedding_manager.generate_embeddings(texts).astype(np.float32)
        norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        self.embeddings = self.embeddings / np.maximum(norms, 1e-12)

    def similarity_search(self, query: str, k: int = 3):
        query_embedding = self.embedding_manager.generate_embeddings([query]).astype(np.float32)[0]
        query_embedding = query_embedding / max(np.linalg.norm(query_embedding), 1e-12)
        scores = self.embeddings @ query_embedding
        top_indices = np.argsort(scores)[::-1][:k]
        return [SearchResult(page_content=self.texts[i], score=float(scores[i])) for i in top_indices]


embedding_manager = EmbeddingManager()
db = SimpleVectorStore(chunks, embedding_manager)

Loading model sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully, embedding dimension: 384
Generating embeddings for 9 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (9, 384)


In [19]:
query = "What are Pushkar's AI skills?"

results = db.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("=" * 50)

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
.com/pushkarsingh26 Career Objective Aspiring AI & Generative AI Engineer with a strong foundation in Python, LLMs, embeddings, vector databases, and RAG pipelines
. Strengths Quick learner with strong adaptability to new and emerging technologies; able to independently explore and implement AI tools with minimal guidance
Pushkar Chhokar +91-7389530419 # pushkarcd123@gmail.com pushkar-chhokar § github


In [38]:
import os
from openai import OpenAI

# -----------------------------
# USER QUERY
# -----------------------------
query = input("Ask about the resume: ")

# -----------------------------
# BETTER RETRIEVAL QUERY
# -----------------------------
search_query = f"""
Resume technical skills, projects,
developer tools, achievements,
education and experience related to:
{query}
"""

# -----------------------------
# RETRIEVE RELEVANT CHUNKS
# -----------------------------
results = db.similarity_search(search_query, k=6)

# CREATE CONTEXT
context = "\n\n".join(
    r.page_content for r in results
)

# -----------------------------
# OPENROUTER CLIENT
# -----------------------------
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("Set OPENROUTER_API_KEY environment variable")

else:

    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )

    # -----------------------------
    # STEP 1 : DOMAIN DETECTION
    # -----------------------------
    domain_prompt = f"""
    Detect the candidate's primary technical domain
    from the resume context.

    Resume Context:
    {context}

    Return ONLY one domain name.

    Possible Domains:
    - GenAI
    - Web Development
    - Data Science
    - Backend Development
    - Cybersecurity
    - DevOps
    - Mobile App Development
    """

    domain_response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are an expert technical recruiter."
            },
            {
                "role": "user",
                "content": domain_prompt
            }
        ]
    )

    domain = domain_response.choices[0].message.content.strip()

    print("\nDetected Domain:", domain)

    # -----------------------------
    # STEP 2 : MAIN ANALYSIS
    # -----------------------------
    final_prompt = f"""
    You are an expert ATS Resume Analyzer.

    Candidate Domain:
    {domain}

    Use ONLY the provided resume context.

    Resume Context:
    {context}

    User Question:
    {query}

    Rules:
    - Do not hallucinate
    - Do not invent skills
    - Be specific
    - If a skill already exists in the resume,
      do NOT mention it as missing
    - Keep answers professional and concise
    """

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"""
                You are an expert ATS resume analyzer
                specialized in {domain}.
                """
            },
            {
                "role": "user",
                "content": final_prompt
            }
        ]
    )

    # -----------------------------
    # FINAL OUTPUT
    # -----------------------------
    print("\n===== AI RESPONSE =====\n")

    print(response.choices[0].message.content)

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)

Detected Domain: GenAI

===== AI RESPONSE =====

Based on the provided resume context, your ATS score appears to be strong for a position related to GenAI, primarily due to the following factors:

1. **Relevant Technical Skills**: You have listed essential programming languages (Python, C, C++), AI/ML & GenAI technologies (LLMs, RAG, embeddings), which are critical for roles in this domain.

2. **Project Experience**: Your involvement in relevant projects, such as IntelliDocs-RAG and DocuChat-RAG, demonstrates practical application of your skills in Generative AI and reflects hands-on experience with tools and frameworks sought by employers.

3. **Education**: You are currently pursuing a B.Tech in Computer Science, which aligns well with the requirements for many entry-level positions in AI and GenAI.

4. **Achievements**: Holding a copyright in generative AI signifies a level of expertise and commitment to the field.

5. **Career Objective**

In [55]:
# =========================================================
# ADVANCED DYNAMIC ATS SCORING SYSTEM (IMPROVED HEURISTICS)
# =========================================================

import re
from collections import Counter

# CLEAN RESUME TEXT
resume_text = text.lower()
lines = [l.strip() for l in resume_text.splitlines() if l.strip()]

# Helper checks
def contains_any(words):
    return any(w in resume_text for w in words)

# REQUIRED SKILLS (use detected_domain if available)
try:
    required_skills = domains[detected_domain]
except Exception:
    required_skills = ["llm", "rag", "langchain", "faiss", "embeddings", "vector database", "chromadb"]

# Expand skill synonyms and categories
core_skills = ["llm", "rag", "langchain", "faiss", "embeddings", "vector database", "chromadb", "transformers", "sentence-transformers"]
related_skills = ["python", "pandas", "numpy", "fastapi", "docker", "streamlit", "sql", "mysql", "postgresql", "git", "github"]
proficiency_tokens = ["expert", "proficient", "strong", "experienced", "familiar with", "familiar"]

# --------------------------------
# TECHNICAL SKILLS (40)
# --------------------------------
technical_max = 40
core_matches = sum(1 for s in core_skills if s in resume_text)
related_matches = sum(1 for s in related_skills if s in resume_text)
# proficiency multiplier if candidate uses strong terms
prof_mult = 1.0 + (0.25 if contains_any(proficiency_tokens) else 0.0)
# raw technical score from matches (weighted)
raw_tech = (core_matches * 2 + related_matches * 1) * prof_mult
# normalize to technical_max using a reasonable cap
technical_score = min(technical_max, int(round(raw_tech * (technical_max / max(1, (len(core_skills)*2 + len(related_skills)))))))

# --------------------------------
# PROJECTS (15)
# --------------------------------
project_max = 15
# detect explicit project sections or bullet items mentioning 'project' or presence of 'project:' headings
project_section_count = len(re.findall(r"\bprojects?\b", resume_text))
# count lines that look like project bullets (contain project keywords or techs and hyphen/bullet)
project_bullets = sum(1 for l in lines if ("project" in l or any(t in l for t in core_skills+related_skills)) and (l.startswith("-") or l.startswith("*") or l[0].isdigit()))
project_score = 0
if project_section_count > 0 or project_bullets > 0:
    project_score += 4
project_score += min(6, project_bullets)
# metrics present
if re.search(r"\d+%|\d+\s*\+|\d+\s+years|\b\d{1,2}\b\s+months", resume_text):
    project_score += 3
# deployment evidence
if contains_any(["deployed", "deployment", "hosted", "live", "hugging face", "aws", "gcp", "azure"]):
    project_score += 1
project_score = min(project_max, project_score)

# --------------------------------
# EXPERIENCE (10)
# --------------------------------
experience_max = 10
experience_score = 0
# detect explicit years of experience (try several patterns)
years = 0
m = re.findall(r"(\d+)\s+years", resume_text)
if m:
    years = max(int(x) for x in m)
else:
    m2 = re.findall(r"experience\s*[:\-]?\s*(\d+)\s+years", resume_text)
    if m2:
        years = max(int(x) for x in m2)
# score mapping
if years >= 4:
    experience_score = 10
elif years >= 2:
    experience_score = 7
elif years >= 1:
    experience_score = 5
elif contains_any(["internship", "intern"]):
    experience_score = 5
else:
    experience_score = 0

# --------------------------------
# ACHIEVEMENTS (10)
# --------------------------------
ach_max = 10
ach_keywords = ["award", "winner", "certification", "published", "paper", "patent", "copyright", "recognition"]
ach_count = sum(1 for k in ach_keywords if k in resume_text)
ach_score = min(ach_max, ach_count * 2)
# check for high-value items
if "patent" in resume_text or "copyright" in resume_text or "intellectual property" in resume_text:
    ach_score = min(ach_max, ach_score + 2)

# --------------------------------
# TOOLS & PLATFORMS (15)
# --------------------------------
tools = ["docker", "git", "github", "streamlit", "fastapi", "mysql", "faiss", "hugging face", "postgresql", "aws", "gcp", "azure"]
tool_matches = [t for t in tools if t in resume_text]
# prefer unique tools count
tool_score = min(15, len(tool_matches) * 2)
# if cloud presence + git present, small boost
if any(c in resume_text for c in ["aws", "gcp", "azure"]) and ("git" in resume_text or "github" in resume_text):
    tool_score = min(15, tool_score + 1)

# --------------------------------
# EDUCATION (5)
# --------------------------------
edu_score = 0
if contains_any(["b.tech", "bachelor", "b.sc", "m.tech", "master"]):
    edu_score += 3
if "cgpa" in resume_text or re.search(r"\b\d\.\d{1,2}\b", resume_text):
    edu_score += 2
edu_score = min(5, edu_score)

# --------------------------------
# RESUME STRUCTURE (5)
# --------------------------------
structure_score = 0
sections = ["education", "projects", "skills", "achievements", "experience", "contact"]
structure_score += sum(1 for s in sections if s in resume_text)
# density: count bullets
bullets = sum(1 for l in lines if l.startswith("-") or l.startswith("*"))
if bullets > 5:
    structure_score += 1
structure_score = min(5, structure_score)

# RAW BREAKDOWN
raw_breakdown = {
    "Technical Skills": technical_score,
    "Projects": project_score,
    "Experience": experience_score,
    "Achievements": ach_score,
    "Tools & Platforms": tool_score,
    "Education": edu_score,
    "Resume Structure": structure_score,
}
raw_total = sum(raw_breakdown.values())

# FINAL OUTPUT (RAW)
print(f"\nRAW TOTAL: {raw_total}/100")
print("\n" + "=" * 60)
print("ATS SCORE BREAKDOWN (RAW - improved heuristics)")
print("=" * 60)
for category, score in raw_breakdown.items():
    totals = {"Technical Skills": 40, "Projects": 15, "Experience": 10, "Achievements": 10, "Tools & Platforms": 15, "Education": 5, "Resume Structure": 5}
    total = totals.get(category, 5)
    print(f"{category}: {score}/{total}")

print("\n" + "=" * 60)
print("MISSING SKILLS")
print("=" * 60)
missing_skills = [s for s in required_skills if s.lower() not in resume_text]
if missing_skills:
    for skill in missing_skills:
        print("-", skill)
else:
    print("No major missing skills detected.")

# Improvement suggestions (based on improved heuristics)
print("\n" + "=" * 60)
print("IMPROVEMENT SUGGESTIONS")
print("=" * 60)
if raw_breakdown["Experience"] < 5:
    print("- Add internship or real-world experience.")
if raw_breakdown["Projects"] < 10:
    print("- Add more impactful projects with measurable results and deployment evidence.")
if raw_breakdown["Technical Skills"] < 30:
    print("- Include more domain-specific keywords and proficiency indicators (e.g., 'proficient', 'expert').")
if raw_breakdown["Tools & Platforms"] < 10:
    print("- List deployment and cloud tools (AWS/GCP/Azure) and version control (git).")
if missing_skills:
    print("- Missing important skills:")
    for skill in missing_skills:
        print(f"  • {skill}")
if not any([raw_breakdown["Experience"] < 5, raw_breakdown["Projects"] < 10, raw_breakdown["Technical Skills"] < 30, missing_skills]):
    print("- Resume is well optimized for the detected domain.")


RAW TOTAL: 82/100

ATS SCORE BREAKDOWN (RAW - improved heuristics)
Technical Skills: 40/40
Projects: 8/15
Experience: 5/10
Achievements: 4/10
Tools & Platforms: 15/15
Education: 5/5
Resume Structure: 5/5

MISSING SKILLS
No major missing skills detected.

IMPROVEMENT SUGGESTIONS
- Add more impactful projects with measurable results and deployment evidence.
